# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset is sourced from a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}\n")
print(f"Authors: {metadata.author}")
print(f"Date Published: {metadata.datePublished}")
print(f"Keywords: {metadata.keywords}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Below, we will identify all record sets provided by the dataset, and for a selected record set, we will list out its fields (columns) and their `@id`s.

In [ ]:
# List all available record sets and their details
record_sets = dataset.metadata.recordSets

if not record_sets:
    print("No record sets found in the dataset metadata. Attempting to read from main distribution file...")
else:
    for record_set in record_sets:
        print(f"RecordSet: {record_set['@id']}")
        print(f"  Name: {record_set.get('name', '')}")
        print(f"  Description: {record_set.get('description', '')}")
        print(f"  Fields: {[f['@id'] for f in record_set.get('fields', [])]}")
        print()

In [ ]:
# Discover the record set identifiers
# If no recordSets are present in the metadata, attempt to use main distribution file as a record set.

# The Croissant schema defines the distribution file, we'll use its @id
distribution_list = dataset.metadata.distribution
if isinstance(distribution_list, dict):
    distribution_list = [distribution_list]

main_recordset_id = distribution_list[0]['@id'] if distribution_list else None
print(f"Main data file (record set) @id: {main_recordset_id}")

Let's print a few rows from the main record set using its `@id`. This helps reveal the field `@id`s used as DataFrame columns.

In [ ]:
# Print sample records and field names from the main record set
num_print = 3
rows = []
for i, record in enumerate(dataset.records(record_set=main_recordset_id)):
    if i < num_print:
        print(json.dumps(record, indent=2)[:1000])  # Print first 1000 chars for brevity
        rows.append(record)
    else:
        break
# Show field ids (column names) from a sample row
if rows:
    print("\nFields (`@id`s as DataFrame columns):\n", list(rows[0].keys()))

## 3. Data Extraction
Load data from the discovered record set into a DataFrame for analysis. Use the record set and field `@id` values as identified above.

In [ ]:
# Extract data from the main distribution record set
selected_record_sets = [main_recordset_id]
dataframes = {}

for record_set_id in selected_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    print(f"Loaded {len(records)} records for record set {record_set_id}.")
    if records:
        print(f"Columns/fields for {record_set_id}:\n{df.columns.tolist()}")
        display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, or grouping data by key attributes for preliminary analysis.

**Note:** We need to reference fields by their `@id`. Take a look at the available numeric and categorical field `@id`s in the DataFrame.

In [ ]:
# Preview numeric fields from the columns
df = dataframes.get(main_recordset_id)
if df is not None:
    # Print columns and guess field types
    print("Field IDs / columns:", df.columns.tolist())
    # Try to infer numeric fields
    sample_row = df.iloc[0].to_dict()
    candidate_numeric_fields = [col for col in df.columns
                                if pd.api.types.is_numeric_dtype(df[col]) or
                                   (isinstance(sample_row.get(col), (int, float)))]
    print("Candidate numeric fields:", candidate_numeric_fields)
    # If nothing detected, suggest user to supply or pick any numeric field
    chosen_numeric_field = candidate_numeric_fields[0] if candidate_numeric_fields else df.columns[0]  # pick first field as default
    print(f"\nSelected numeric field for illustration: {chosen_numeric_field}\n")
    # Filter: e.g., keep values above the mean
    threshold = df[chosen_numeric_field].mean() if pd.api.types.is_numeric_dtype(df[chosen_numeric_field]) else 0
    filtered_df = df[df[chosen_numeric_field] > threshold]
    print(f"Filtered records with {chosen_numeric_field} > {threshold:.3g} (mean value): {len(filtered_df)} records")
    # Normalize the field
    filtered_df[chosen_numeric_field + '_normalized'] = (
        (filtered_df[chosen_numeric_field] - filtered_df[chosen_numeric_field].mean()) /
        filtered_df[chosen_numeric_field].std()
    )
    print(filtered_df[[chosen_numeric_field, chosen_numeric_field + '_normalized']].head())
    # Try grouping if a likely categorical field exists
    candidate_group_fields = [col for col in df.columns if df[col].dtype == 'object' and col != chosen_numeric_field]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[chosen_numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean of {chosen_numeric_field}):")
        print(grouped_df.head())
else:
    print("No records found in data extraction.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Let's display a histogram of the selected numeric field, a box plot, and a bar plot of the group means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None:
    # Histogram
    plt.figure(figsize=(7,4))
    sns.histplot(df[chosen_numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {chosen_numeric_field}")
    plt.xlabel(chosen_numeric_field)
    plt.ylabel("Count")
    plt.show()
    # Boxplot
    plt.figure(figsize=(6,1.5))
    sns.boxplot(x=df[chosen_numeric_field])
    plt.title(f"Boxplot of {chosen_numeric_field}")
    plt.show()
    # If group_field is available, plot means per group
    if 'group_field' in locals() and group_field:
        group_means = df.groupby(group_field)[chosen_numeric_field].mean().sort_values(ascending=False).head(10)
        group_means.plot(kind='bar', figsize=(9,4), title=f"Top 10 {group_field} by mean {chosen_numeric_field}")
        plt.ylabel(f"Mean {chosen_numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to programmatically explore the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using `mlcroissant` by referencing all major entities with their `@id`. We loaded metadata, extracted and inspected the dataset, identified and analyzed numeric fields, performed some basic filtering and normalization, and visualized the data distributions.

For more detailed analysis, consult the full data schema and description at the [Sen Science FAIR² dataset page](https://sen.science/doi/10.71728/senscience.vq4a-28xa).